# PhysIR — DISTS per-image evaluation (Kaggle, ONE FLOW, resumable)

Adds **DISTS** (Deep Image Structure and Texture Similarity, Ding et al. 2020) as an independent
full-reference perceptual metric across the 12 models that carry the thesis's comparisons.

**Why.** The perceptual axis currently rests on LPIPS — which is *also the loss the comparison
arm trains on*. "Physics beats the perceptual loss on LPIPS itself" is therefore open to the
objection that the metric is kin to the objective. DISTS is not optimised by any arm, and scores
images by **structure/texture correlation** rather than learned feature-difference weights.

⚠️ **Scope the independence claim honestly:** DISTS shares LPIPS's **VGG16 backbone**. What is
independent is (i) the training objective — nothing here was trained on DISTS — and (ii) the
comparison function. It is *not* an independent feature space, and Chapter 4 should say so.

⚠️ **Input convention — opposite to LPIPS.** pyiqa's DISTS normalises internally with ImageNet
statistics, so it takes **[0,1]** tensors. LPIPS wants **[−1,1]**. Passing `pred*2−1` here would
be silently wrong. A startup self-check asserts `DISTS(x, x) ≈ 0` before any model is scored.

## The 12 targets (scope b)
- **384 anchors (4):** PhysIR, baseline, perceptual arm, released DarkIR-m
- **256 ablation arms @last (8):** baseline, phase, retinex, full, blur, blur-instead-of-edge,
  perceptual, full+perceptual

Seed replicates are deliberately excluded — robustness is already established on three metrics
across three seeds (Ch.3-guide G12); a fourth metric adds little there.

## Inputs to attach
| # | dataset | for |
|---|---|---|
| 1 | `physir-code` | model builder + loader (needs the `shuffle=False` fix; guard enforces it) |
| 2 | `lolblur-dataset` | the 1800-image test set |
| 3 | `gabrielafotey/physir-models-evaluations-set` | the four 384 checkpoints |
| 4–6 | **Add Input → Your Work →** `stage5_set1`, `stage5_set2`, `stage5_set3` | the eight 256 checkpoints |

**Accelerator: GPU T4 × 2.** ~15–30 min per model → **~4–6 h total**. Fully resumable: finished
models are skipped, and a model interrupted mid-way resumes from its last flushed row, so a
session cut costs at most a few hundred images.

**Output:** `per_image_dists/per_image_dists_<name>.csv` with `idx, fname, dists` — joinable to
the existing per-image CSVs on `idx`, with `fname` for verification. Provenance (pyiqa version,
input convention, `DISTS(x,x)` self-check) is written to `dists_provenance.json`.

## 1. Install

In [ ]:
# Record torch BEFORE installing: pyiqa drags in timm/scipy/opencv and pip is free to
# swap the CUDA torch wheel for a CPU one to satisfy them. That would not raise — it would
# just silently run the whole eval on CPU (or fail at .cuda()), so we pin and verify.
import torch as _t0
_TORCH_BEFORE, _CUDA_BEFORE = _t0.__version__, _t0.cuda.is_available()
print(f'before: torch {_TORCH_BEFORE}, cuda {_CUDA_BEFORE}')

# pyiqa provides DISTS; ptflops/einops/pytorch_msssim are UNGUARDED imports pulled in by
# `import train` and `from losses import ...`. pandas/PyYAML/matplotlib ship with the image.
!pip install -q pyiqa pytorch-msssim lpips einops ptflops tabulate

# Fail-fast: check every needed import BEFORE any model or data is touched, so a bad
# install costs seconds rather than surfacing partway into a multi-hour session.
import importlib.util as _iu
_need = ['pyiqa', 'ptflops','lpips', 'einops', 'pytorch_msssim', 'torch', 'torchvision', 'numpy', 'PIL']
_missing = [m for m in _need if _iu.find_spec(m) is None]
print('dependency check:', 'all present ✅' if not _missing else f'MISSING {_missing} ❌')
assert not _missing, f'install failed for {_missing} — re-run this cell (Internet must be ON)'

# torch must be untouched. If this trips, RESTART the kernel and reinstall the pinned
# torch version first; do not continue, the numbers would not be comparable.
import subprocess, sys
_after = subprocess.run([sys.executable, '-c',
    'import torch; print(torch.__version__, torch.cuda.is_available())'],
    capture_output=True, text=True).stdout.split()
print(f'after:  torch {_after[0]}, cuda {_after[1]}')
assert _after[0] == _TORCH_BEFORE, (
    f'pip changed torch {_TORCH_BEFORE} -> {_after[0]} — restart kernel and pin torch')
assert _after[1] == str(_CUDA_BEFORE), (
    f'CUDA availability changed {_CUDA_BEFORE} -> {_after[1]} — a CPU-only wheel was installed')
print('torch unchanged ✅')

## 2. Paths + hard guard (refuses stale physir-code)

In [ ]:
import os, sys, glob, re, json, shutil
import numpy as np, torch

INPUT_BASE = '/kaggle/input'
WORK = '/kaggle/working'
PERIMG = os.path.join(WORK, 'per_image_dists');           os.makedirs(PERIMG, exist_ok=True)
STRIP  = os.path.join(WORK, 'checkpoints_stripped'); os.makedirs(STRIP, exist_ok=True)

ROOTS = sorted(d for d in glob.glob(os.path.join(INPUT_BASE,'*')) if os.path.isdir(d))
def _dir_containing(marker, hint=None):
    ordered = sorted(ROOTS, key=lambda d: (hint or '') not in os.path.basename(d))
    for r in ordered:
        if os.path.exists(os.path.join(r, marker)): return r
        hits = sorted(glob.glob(os.path.join(r,'**',marker), recursive=True))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep) or r
    return None

CODE_ROOT = _dir_containing('train.py', 'physir-code')
LOLBLUR   = _dir_containing(os.path.join('test','low_blur_noise'), 'lolblur')
assert CODE_ROOT and LOLBLUR, f'code/data not found among {ROOTS}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── HARD GUARD: the uploaded loader must NOT shuffle the test set ─────────────
_ds = open(os.path.join(CODE_ROOT,'data','dataset_reader','dataset_LOLBlur.py')).read()
_bad = [d for d in re.findall(r'DataLoader\([^)]*\)', _ds, re.S)
        if 'shuffle=True' in d.replace(' ', '')]
assert not _bad, ('STALE physir-code: the test DataLoader still has shuffle=True. '
                  'Re-upload physir-code with the shuffle=False fix before running.')
print('✅ guard passed — uploaded loader does not shuffle the test set')

def find_ckpt(fname):
    # Checkpoints live in `physir-models-evaluations-set` (the s1337 pair was appended to it),
    # but the search is name-based across every attached input at any depth, so the dataset
    # layout is irrelevant — nothing breaks if files move or the slug changes.
    ordered = sorted(ROOTS, key=lambda r: 'models' not in os.path.basename(r).lower())
    for r in ordered:
        for h in sorted(glob.glob(os.path.join(r, '**', fname), recursive=True)):
            if h.endswith('.pt'): return h
    return None
print('attached:', [os.path.basename(r) for r in ROOTS])

## 2b. Session restore (attach the previous version's output to resume)

In [ ]:
# ── Session restore: carry finished / partly-finished CSVs back into the working dir ──
# Continuation sessions: Add Input → Your Work → this notebook (previous version's output).
# Completed arms are then skipped; a partly-written CSV resumes mid-arm (see the eval loop).
import shutil
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/per_image_dists/*.csv', '/kaggle/input/*/*/per_image_dists/*.csv'):
        for src in glob.glob(pat):
            dst = os.path.join(PERIMG, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst); n += 1
    if n:
        print(f'restored {n} CSV(s) from a previous session:')
        for f in sorted(os.listdir(PERIMG)):
            rows = sum(1 for _ in open(os.path.join(PERIMG, f))) - 1
            print(f'    {f}  {rows}/1800 rows' + ('  (complete)' if rows >= 1800 else '  → will resume'))
    else:
        print('fresh session — nothing to restore')
_restore()

## 3. Targets + checkpoint availability

In [ ]:
# name → checkpoint filename.  Names match the existing per-image CSVs so the DISTS column
# can be joined on idx without any renaming.
TARGETS = [
  # ── 384 anchors ──
  ('stage4_physir_final', 'stage4_physir_final_latest.pt'),
  ('stage5_baseline',     'stage5_baseline_latest.pt'),
  ('stage5_percep',       'stage5_percep_latest.pt'),
  ('darkir_m_released',   'DarkIR_384.pt'),
  # ── 256 ablation arms @last (seed 42) ──
] + [(f'{a}_last', f'{a}_latest.pt') for a in
     ['abl_baseline','abl_phase','abl_retinex','abl_full',
      'abl_blur','abl_blurnoedge','abl_percep','abl_full_percep']]

ALIASES = {}

print(f'{"target":24s} {"checkpoint":34s} status')
missing = []
for name, fn in TARGETS:
    p = find_ckpt(fn)
    out = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
    st = 'DONE ✅' if os.path.exists(out) else (
        'found: ' + os.path.relpath(p, INPUT_BASE) if p else 'CKPT MISSING ❌')
    if not p and not os.path.exists(out): missing.append(fn)
    print(f'{name:24s} {fn:34s} {st}')
if missing:
    print('\n⚠ missing checkpoints — attach the matching set-notebook output:')
    for m in missing: print('   ', m)
print(f'\n{len(TARGETS)} targets · aliases: {ALIASES}')

## 4. Eval engine (batched, per-sample metrics, filename column)

In [ ]:
import csv, time
import train as T
from data import create_test_data
from pytorch_msssim import ssim as _ssim
import lpips as _lpips_mod

BATCH = 4          # uniform 1120x640 test images → batching is safe and ~3x faster
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# The DarkIR-convention LPIPS column exists only for the cross-paper comparison; the seed
# replicates never use it. Skipping it on CPU halves the dominant cost (two VGG passes -> one).
DUAL_LPIPS = (dev == 'cuda')
print('device:', dev, '| batch', BATCH,
      '| dual-LPIPS', 'on' if DUAL_LPIPS else 'OFF (CPU: not needed for replicate arms)')
if dev != 'cuda':
    torch.set_num_threads(os.cpu_count() or 4)
    print(f'  CPU run: {os.cpu_count()} threads. Expect ~3-6 h per arm; the loop is idempotent,')
    print('  so a cut session resumes from the next unfinished CSV.')

opt_test = {'name':'LOLBlur',
            'val':{'test_path': os.path.join(LOLBLUR,'test'), 'batch_size_test': BATCH},
            'train':{'n_workers':2, 'verbose':False}}
raw,_ = create_test_data(rank=0, world_size=1, opt=opt_test)
LOADER = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']
DS = LOADER.dataset
# clip-qualified so the key is UNIQUE: LOLBlur test basenames repeat across its 30 clips
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p))
          for p in DS.imgs_low]                          # dataset sorts its lists
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
print(f'test set: {len(DS)} images | first: {FNAMES[0]} | last: {FNAMES[-1]}')

# runtime proof that iteration order == sorted order (i.e. idx is a stable image id)
_h, _l = next(iter(LOADER))
assert len(FNAMES) == len(DS), 'filename list length mismatch'
print('✅ order check: loader is non-shuffled, idx ↔ sorted filename is stable')

# (LPIPS is not used here — this notebook computes DISTS only.)
NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
       'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

def load_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
    assert st is not None, f'no weights in {path}'
    m = T.build_model(NET).to(dev)
    m.load_state_dict({k.replace('module.',''):v for k,v in st.items()}, strict=False)
    return m.eval(), ck

# ── DISTS setup + the convention self-check ─────────────────────────────────
import pyiqa, json as _json
DISTS = pyiqa.create_metric('dists', device=dev, as_loss=False)
print('pyiqa', pyiqa.__version__, '| dists lower_better =', DISTS.lower_better)

# ── Guards that actually test something ────────────────────────────────────
# NOTE: "DISTS(x,x) ~ 0" is NOT a convention check — identical inputs score 0 at ANY input
# range, so that assert would pass even if we wrongly fed [-1,1] tensors. The real protections
# are (a) pyiqa's own check_input_range, verified live below, and (b) the fact that `pred` is
# clamp(0,1) and `high` comes from ToTensor, both already in [0,1].
_x = torch.rand(2, 3, 128, 128, device=dev)
_y = torch.rand_like(_x)

# (i) metric is alive and discriminating, and returns ONE SCORE PER IMAGE (not a batch mean)
_same, _diff = DISTS(_x, _x).flatten(), DISTS(_x, _y).flatten()
assert _same.numel() == 2 and _diff.numel() == 2, \
    f'DISTS returned {_diff.numel()} score(s) for a batch of 2 — per-sample output required'
assert float(_same.mean()) < 1e-3, 'DISTS(x,x) is not ~0 — metric is misconfigured'
assert float(_diff.mean()) > float(_same.mean()), 'DISTS is not discriminating'

# (ii) the range guard is ACTIVE: feeding LPIPS-style [-1,1] must raise, not silently score.
_range_guard = False
try:
    DISTS(_x * 2 - 1, _y * 2 - 1)
except AssertionError:
    _range_guard = True
assert _range_guard, ('pyiqa check_input_range is OFF — a wrong input range would score '
                      'silently. Re-create the metric with check_input_range=True.')
print(f'self-check: per-sample ✅ | DISTS(x,x)={float(_same.mean()):.6f} | '
      f'DISTS(x,y)={float(_diff.mean()):.4f} | [0,1] range guard ACTIVE ✅')
PROV = {'metric': 'DISTS', 'impl': f'pyiqa {pyiqa.__version__} (create_metric("dists"))',
        'backbone': 'VGG16 (shared with LPIPS-vgg — not an independent feature space)',
        'input_range': '[0,1] (ImageNet normalisation applied internally by pyiqa)',
        'lower_better': bool(DISTS.lower_better),
        'check_input_range': True,
        'resizing': 'none — scored at native 1120x640',
        'per_sample_output': True,
        'self_check_identical': float(_same.mean()),
        'self_check_random': float(_diff.mean())}

HEADER_COLS = ['idx', 'fname', 'dists']

def _rows_done(path):
    """Completed rows, truncated to a whole-batch boundary so a half-written batch is redone."""
    if not os.path.exists(path): return 0
    rows = list(csv.reader(open(path)))[1:]
    done = (len(rows) // BATCH) * BATCH
    if done != len(rows):
        with open(path, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(HEADER_COLS); w.writerows(rows[:done])
    return done

@torch.no_grad()
def eval_dists(model, out_csv):
    done = _rows_done(out_csv)
    if done: print(f'    resuming at {done}/{len(DS)}')
    fh = open(out_csv, 'a' if done else 'w', newline=''); w = csv.writer(fh)
    if not done: w.writerow(HEADER_COLS); fh.flush()
    i0, t0 = 0, time.time()
    for high, low in LOADER:
        n = high.shape[0]
        if i0 + n <= done:
            i0 += n; continue
        high, low = high.to(dev, non_blocking=True), low.to(dev, non_blocking=True)
        pred = model(low, side_loss=False).clamp(0, 1)[:, :, :high.shape[2], :high.shape[3]]
        if i0 == done:   # once per model: prove the real tensors are in the expected range
            assert 0.0 <= float(high.min()) and float(high.max()) <= 1.0, 'GT not in [0,1]'
            assert 0.0 <= float(pred.min()) and float(pred.max()) <= 1.0, 'prediction not in [0,1]'
        d = DISTS(pred, high).flatten()          # [0,1] inputs — see the convention note above
        for k in range(n):
            w.writerow([i0 + k, FNAMES[i0 + k], float(d[k])])
        i0 += n
        if i0 % 200 < BATCH:
            fh.flush(); os.fsync(fh.fileno())
            r = (time.time() - t0) / max(i0 - done, 1)
            print(f'    {i0}/{len(DS)}  ~{r * (len(DS) - i0) / 60:.1f} min left', flush=True)
    fh.close()
    vals = [float(r[2]) for r in list(csv.reader(open(out_csv)))[1:]]
    return sum(vals) / len(vals), len(vals)


## 5. RUN — 12 evals, idempotent

In [ ]:
for name, fn in TARGETS:
    out = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
    if _rows_done(out) >= len(DS):
        print(f'✅ {name}: complete — skip'); continue
    p = find_ckpt(fn)
    if not p:
        print(f'⏭  {name}: checkpoint {fn} not attached — skip'); continue
    print(f'\n=== DISTS {name}  ({os.path.relpath(p, INPUT_BASE)}) ===', flush=True)
    model, _ = load_model(p)
    mean_d, n = eval_dists(model, out)
    print(f'  → DISTS {mean_d:.4f}  (n={n}, lower is better)')
    del model; torch.cuda.empty_cache()

_json.dump(PROV, open(os.path.join(WORK, 'dists_provenance.json'), 'w'), indent=2)
print('\nDISTS means (lower = better):')
for name, _ in TARGETS:
    f = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
    if _rows_done(f) >= len(DS):
        v = [float(r[2]) for r in list(csv.reader(open(f)))[1:]]
        print(f'  {name:24s} {sum(v)/len(v):.4f}')


## 6. Verify pairing is now real + bundle

## 6. Generalisation DISTS (LOLv2 + LSRW) — guarded, resumable, optional

In [ ]:
# ══ GENERALISATION DISTS — configuration + PRE-RUN AVAILABILITY GUARD ══════════
# Runs BEFORE any eval so a missing dataset is known in ~2 s, not after the LOLBlur pass.
# Nothing here raises: a dataset that cannot be found is recorded as SKIPPED and framed in
# Ch.4 (Log #28) rather than silently vanishing from the table.
#
# ┌─ EDIT HERE if auto-discovery fails ────────────────────────────────────────┐
# │ Paste an absolute /kaggle/input/... path to force a dataset. None = auto.  │
MANUAL_PATHS = {
    'lolv2': None,   # expects <path>/LOL-v2/{Real_captured,Synthetic}/test/{Low,Normal}
    'lsrw':  None,   # expects <path>/**/Eval/{Huawei,Nikon}/{low,high}
}
# └────────────────────────────────────────────────────────────────────────────┘
# Kaggle slugs (confirmed from stage4_final_colab / stage5_*_colab):
#   LOLv2 → tanhyml/lol-v2-dataset      LSRW → bjzh123/lsrw123
# Add Input → search the slug → attach. Real-LOLBlur is deliberately NOT here: DISTS is
# full-reference and Real-LOLBlur has no ground truth (Log #26).
SLUG_HINT = {'lolv2': 'tanhyml/lol-v2-dataset', 'lsrw': 'bjzh123/lsrw123'}

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp'}
def _sub(d, names):
    for c in os.listdir(d):
        if c.lower() in names and os.path.isdir(os.path.join(d, c)): return os.path.join(d, c)
    return None

def _discover_lolv2(root):
    """<root>/**/LOL-v2/{Real_captured,Synthetic}/test/{Low,Normal} → [(label, low, high)]"""
    out = []
    for base, dirs, _ in os.walk(root):
        if os.path.basename(base).lower() != 'lol-v2': continue
        for variant in ('Real_captured', 'Synthetic'):
            v = _sub(base, {variant.lower()})
            if not v: continue
            t = _sub(v, {'test'}) or v
            low, high = _sub(t, {'low'}), _sub(t, {'normal', 'high'})
            if low and high: out.append((f'LOLv2_{variant}', low, high))
        if out: break
    return out

def _discover_lsrw(root):
    """Eval Huawei/Nikon dirs holding low/ + high/, excluding the Training split."""
    out = []
    for base, dirs, _ in os.walk(root):
        if 'training' in base.lower() or 'eval' not in base.lower(): continue
        low, high = _sub(base, {'low'}), _sub(base, {'high', 'normal'})
        if low and high: out.append((f'LSRW_{os.path.basename(base)}', low, high))
    return sorted(set(out))

GEN_SETS, GEN_SKIPPED = [], []
for key, disc in (('lolv2', _discover_lolv2), ('lsrw', _discover_lsrw)):
    roots = [MANUAL_PATHS[key]] if MANUAL_PATHS[key] else \
            [r for r in ROOTS if key.replace('v2', '') in os.path.basename(r).lower()] or ROOTS
    found = []
    for r in roots:
        if r and os.path.isdir(r):
            found = disc(r)
            if found: break
    if found:
        GEN_SETS += found
    else:
        GEN_SKIPPED.append(key)

print('generalisation dataset availability')
print('-' * 72)
for label, low, high in GEN_SETS:
    n_lo = len([f for f in os.listdir(low) if os.path.splitext(f)[1].lower() in IMG_EXT])
    n_hi = len([f for f in os.listdir(high) if os.path.splitext(f)[1].lower() in IMG_EXT])
    ok = '✅' if n_lo == n_hi and n_lo else '⚠ PAIR MISMATCH'
    print(f'  {label:22s} {n_lo:4d} pairs  {ok}   {os.path.relpath(low, INPUT_BASE)}')
for key in GEN_SKIPPED:
    print(f'  {key:22s}   ❌ NOT FOUND — attach `{SLUG_HINT[key]}` '
          f'(Add Input → Datasets → search the slug),')
    print(f'  {"":22s}      or set MANUAL_PATHS[\'{key}\'] above to its /kaggle/input path.')
print('-' * 72)
print(f'{len(GEN_SETS)} dataset(s) will be evaluated · {len(GEN_SKIPPED)} unavailable')
print('Real-LOLBlur: permanently excluded — DISTS is full-reference, that set has no GT.')
if not GEN_SETS:
    print('\n⚠ No generalisation data attached. The LOLBlur DISTS results are UNAFFECTED;')
    print('  this stage simply produces nothing and Ch.4 Log #28 frames the gap.')

In [ ]:
# ══ GENERALISATION DISTS EVAL — 3 arms x available datasets, resumable ════════
# Three arms only, matching the existing generalisation table (PhysIR / baseline / perceptual).
# DarkIR-m is absent from that table, so adding it to the DISTS column alone would make the
# row non-parallel — the exact asymmetry this stage exists to remove.
import torchvision.transforms.functional as TF
from PIL import Image

GEN_ARMS = [('stage4_physir_final', 'stage4_physir_final_latest.pt'),
            ('stage5_baseline',     'stage5_baseline_latest.pt'),
            ('stage5_percep',       'stage5_percep_latest.pt')]
GENDIR = os.path.join(WORK, 'per_image_dists_gen'); os.makedirs(GENDIR, exist_ok=True)

def _pad8(x):
    _, _, h, w = x.shape
    ph, pw = (8 - h % 8) % 8, (8 - w % 8) % 8
    return (torch.nn.functional.pad(x, (0, pw, 0, ph), mode='reflect') if ph or pw else x), h, w

@torch.no_grad()
def eval_gen(model, low_dir, high_dir, out_csv):
    lo = sorted(f for f in os.listdir(low_dir)  if os.path.splitext(f)[1].lower() in IMG_EXT)
    hi = sorted(f for f in os.listdir(high_dir) if os.path.splitext(f)[1].lower() in IMG_EXT)
    if len(lo) != len(hi) or not lo:
        print(f'    ⚠ pair mismatch ({len(lo)} vs {len(hi)}) — skipped'); return None, 0
    done = _rows_done_n(out_csv)
    fh = open(out_csv, 'a' if done else 'w', newline=''); w = csv.writer(fh)
    if not done: w.writerow(['idx', 'fname', 'dists']); fh.flush()
    for i, (lf, hf) in enumerate(zip(lo, hi)):
        if i < done: continue
        low  = TF.to_tensor(Image.open(os.path.join(low_dir,  lf)).convert('RGB')).unsqueeze(0).to(dev)
        high = TF.to_tensor(Image.open(os.path.join(high_dir, hf)).convert('RGB')).unsqueeze(0).to(dev)
        xin, H, W = _pad8(low)
        pred = model(xin, side_loss=False).clamp(0, 1)[:, :, :H, :W]
        Hm, Wm = min(H, high.shape[2]), min(W, high.shape[3])
        d = DISTS(pred[:, :, :Hm, :Wm], high[:, :, :Hm, :Wm]).flatten()   # [0,1] convention
        w.writerow([i, lf, float(d[0])]); fh.flush()
    fh.close()
    v = [float(r[2]) for r in list(csv.reader(open(out_csv)))[1:]]
    return (sum(v) / len(v) if v else None), len(v)

def _rows_done_n(path):
    return 0 if not os.path.exists(path) else max(0, sum(1 for _ in open(path)) - 1)

if GEN_SETS:
    for arm, fn in GEN_ARMS:
        p = find_ckpt(fn)
        if not p:
            print(f'⏭  {arm}: checkpoint not attached — skip'); continue
        model = None
        for label, low, high in GEN_SETS:
            out = os.path.join(GENDIR, f'per_image_dists_{label}_{arm}.csv')
            n_tot = len([f for f in os.listdir(low) if os.path.splitext(f)[1].lower() in IMG_EXT])
            if _rows_done_n(out) >= n_tot and n_tot:
                print(f'✅ {label}/{arm}: complete — skip'); continue
            if model is None:
                model, _ = load_model(p)
            print(f'=== DISTS {label} / {arm} ===', flush=True)
            mean_d, n = eval_gen(model, low, high, out)
            if mean_d is not None:
                print(f'  → DISTS {mean_d:.4f}  (n={n}, lower is better)')
        if model is not None:
            del model; torch.cuda.empty_cache()

    print('\ngeneralisation DISTS means (lower = better):')
    for label, _, _ in GEN_SETS:
        row = f'  {label:22s}'
        for arm, _ in GEN_ARMS:
            f = os.path.join(GENDIR, f'per_image_dists_{label}_{arm}.csv')
            if os.path.exists(f):
                v = [float(r[2]) for r in list(csv.reader(open(f)))[1:]]
                row += f'  {arm.replace("stage4_","").replace("stage5_","")}={sum(v)/len(v):.4f}' if v else ''
        print(row)
else:
    print('no generalisation datasets attached — nothing to do (see the guard above)')

# Coverage manifest: the authoritative record of what DISTS does and does not cover, so
# Ch.4 can state the scope from data rather than memory.
COVERAGE = {'primary': {'LOLBlur_test': True},
            'generalisation': {l: True for l, _, _ in GEN_SETS},
            'unavailable': GEN_SKIPPED,
            'excluded_by_design': {'Real-LOLBlur': 'no ground truth; DISTS is full-reference'},
            'arms_generalisation': [a for a, _ in GEN_ARMS]}
_json.dump(COVERAGE, open(os.path.join(WORK, 'dists_coverage.json'), 'w'), indent=2)
print('\ncoverage manifest → dists_coverage.json')

In [ ]:
# ── Verify the DISTS CSVs are genuinely paired, then bundle ──────────────────
# Structure note: EVERY diagnostic below is optional and wrapped, because the bundling at the
# bottom is the only step that must happen. Two sessions were already lost to a check failing
# after the eval had succeeded; a diagnostic must never be able to strand hours of GPU work.
import csv, glob as _g

def load_d(name):
    p = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
    if not os.path.exists(p): return None
    rows = list(csv.DictReader(open(p)))
    if len(rows) < len(DS): return None                     # partial arm — not comparable yet
    return ([r['fname'] for r in rows], np.array([float(r['dists']) for r in rows]))

try:
    pairs = [('stage4_physir_final','stage5_baseline'), ('stage4_physir_final','darkir_m_released'),
             ('abl_full_last','abl_baseline_last'), ('abl_full_last','abl_percep_last')]
    print(f'{"pair":52s} {"fname match":>12s} {"corr":>7s}  verdict')
    for a, b in pairs:
        A, B = load_d(a), load_d(b)
        if not (A and B): print(f'{a+" vs "+b:52s}  (not both complete)'); continue
        same = A[0] == B[0]
        r = float(np.corrcoef(A[1], B[1])[0, 1])
        print(f'{a+" vs "+b:52s} {str(same):>12s} {r:7.3f}  '
              f'{"PAIRED ✅" if same and r > 0.4 else "PROBLEM ❌"}')
    print('\nExpect fname match = True and corr > 0.4 — image difficulty dominates DISTS as it '
          'does PSNR, so it cancels in the pairing.')
except Exception as e:
    print(f'⚠ pairing check skipped ({type(e).__name__}: {e})')

# Cross-check idx alignment against the existing PSNR/SSIM/LPIPS CSVs, since Ch.4 joins the
# DISTS column onto those on idx. NOTE: legacy CSVs from before the per-image correction have
# NO `fname` column — those are exactly the files that must not be joined, so finding one is a
# finding, not an error. Prefer a 'corrected' path and require the column explicitly.
try:
    cands = _g.glob(os.path.join(INPUT_BASE, '**', 'per_image_stage4_physir_final.csv'),
                    recursive=True)
    cands.sort(key=lambda q: ('corrected' not in q.lower(), q))
    usable = None
    for q in cands:
        cols = (csv.DictReader(open(q)).fieldnames or [])
        tag = 'has fname ✅' if 'fname' in cols else 'LEGACY (no fname) ⛔ do not join'
        print(f'  candidate: {os.path.relpath(q, INPUT_BASE)}  → {tag}')
        if 'fname' in cols and usable is None: usable = q
    B = load_d('stage4_physir_final')
    if usable and B:
        a = [r['fname'] for r in csv.DictReader(open(usable))]
        print('idx/fname join vs existing per-image CSV:',
              '✅ identical' if a == B[0] else '❌ MISMATCH — do not join')
    elif cands and not usable:
        print('⚠ only legacy per-image CSVs are attached — attach the CORRECTED set before '
              'joining DISTS in the tables notebook.')
    else:
        print('(existing per-image CSVs not attached — join verified in the tables notebook)')
except Exception as e:
    print(f'⚠ join cross-check skipped ({type(e).__name__}: {e})')

# ── Bundle: the one step that must always run ────────────────────────────────
done = sorted(f for f in os.listdir(PERIMG)
              if f.endswith('.csv') and sum(1 for _ in open(os.path.join(PERIMG, f))) - 1 >= len(DS))
part = sorted(set(os.listdir(PERIMG)) - set(done))
print(f'\nper_image_dists/: {len(done)} complete, {len(part)} partial')
for f in done: print('   ✅', f)
for f in part: print('   … ', f, '(partial — will resume next session)')

if os.listdir(PERIMG):
    try:
        z = shutil.make_archive(os.path.join(WORK, 'per_image_dists'), 'zip',
                                WORK, os.path.basename(PERIMG))
        print(f'\n✅ archived → {z}  ({os.path.getsize(z)/2**20:.2f} MB)')
    except Exception as e:
        print(f'\n⚠ make_archive failed ({type(e).__name__}: {e}) — writing zip directly')
        import zipfile
        z = os.path.join(WORK, 'per_image_dists.zip')
        with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in os.listdir(PERIMG):
                zf.write(os.path.join(PERIMG, f), os.path.join('per_image_dists', f))
        print(f'✅ archived → {z}')
else:
    print('\n❌ no CSVs produced — see the In[3] status table above.')

for _d, _n in ((GENDIR, 'per_image_dists_gen'),) if 'GENDIR' in dir() else ():
    if os.path.isdir(_d) and os.listdir(_d):
        try:
            _z = shutil.make_archive(os.path.join(WORK, _n), 'zip', WORK, _n)
            print(f'✅ archived → {_z}  ({len(os.listdir(_d))} csv)')
        except Exception as _e:
            print(f'⚠ gen archive failed ({_e})')
print('\nprovenance:', os.path.exists(os.path.join(WORK, 'dists_provenance.json')),
      '| coverage:', os.path.exists(os.path.join(WORK, 'dists_coverage.json')))
print('Save a version, then attach this output to the figures/tables notebook.')